# check kappa results diff

In [4]:
import pandas as pd
import glob
import os
import gzip
import json
from pathlib import Path
from matbench_discovery.enums import MbdKey
from matbench_discovery.metrics.phonons import calculate_kappa_avg

# Define directories
ksrme_our_dir = "/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/logs/ksrme/pet-oam-xl-v1.0.0"

ksrme_pet_dirs = [
    Path("/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/logs/ksrme/pet-oam-xl-v1.0.0-2026-05-21-kappa-103-FIRE-dist=0.03-fmax=0.0001-symprec=1e-05-slice=0_52"),
    Path("/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/logs/ksrme/pet-oam-xl-v1.0.0-2026-05-21-kappa-103-FIRE-dist=0.03-fmax=0.0001-symprec=1e-05-slice=52_103")
]

ksrme_pet_ref_dirs = [
    Path("/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/logs/ksrme/pet-oam-xl-v1.0.0-float32-2026-05-22-kappa-103-FIRE-dist=0.03-fmax=0.0001-symprec=1e-05-slice=0_52"),
    Path("/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/logs/ksrme/pet-oam-xl-v1.0.0-float32-2026-05-22-kappa-103-FIRE-dist=0.03-fmax=0.0001-symprec=1e-05-slice=52_103")
]


In [7]:
from pymatviz.enums import Key
pattern = f"{ksrme_our_dir}/*kappa-103*.json.gz"
file_list = list(glob.glob(pattern))
# Load all results
all_dfs = []
for file_path in file_list:
    with gzip.open(file_path, "rt", encoding="utf-8") as f:
        data = json.load(f)
    data = pd.DataFrame(data)
    all_dfs.append(data)

print(f"\nProcessing...")    
df_ml = pd.concat(all_dfs).reset_index()
if "mat_id" in df_ml.columns:
    df_ml = df_ml.rename(columns={"mat_id": Key.mat_id})
df_ml = df_ml.set_index(Key.mat_id)
df_our = df_ml

from pymatviz.enums import Key
kappa_files = [
    file_path
    for kappa_dir in ksrme_pet_dirs
    for file_path in sorted(kappa_dir.glob("*_kappa.json.gz"))
]
df_kappa = pd.concat([pd.read_json(file_path) for file_path in kappa_files])
df_kappa.index.name = Key.mat_id
df_pet = df_kappa.loc[df_our.index]

from pymatviz.enums import Key
kappa_files = [
    file_path
    for kappa_dir in ksrme_pet_ref_dirs
    for file_path in sorted(kappa_dir.glob("*_kappa.json.gz"))
]
df_kappa = pd.concat([pd.read_json(file_path) for file_path in kappa_files])
df_kappa.index.name = Key.mat_id
df_pet_ref = df_kappa.loc[df_our.index]

df_our[MbdKey.kappa_tot_avg] = df_our[MbdKey.kappa_tot_rta].map(
        calculate_kappa_avg
    )
df_pet[MbdKey.kappa_tot_avg] = df_pet[MbdKey.kappa_tot_rta].map(
        calculate_kappa_avg
    )
df_pet_ref[MbdKey.kappa_tot_avg] = df_pet_ref[MbdKey.kappa_tot_rta].map(
        calculate_kappa_avg
    )


Processing...


In [41]:
df_our[MbdKey.kappa_tot_avg].head()

material_id
mp-1018100    [37.05457813263334]
mp-580941                   [nan]
mp-2133       [44.05179501323334]
mp-1008559    [262.3351661305667]
mp-8880       [73.55557019283333]
Name: kappa_tot_avg, dtype: object

In [42]:
df_pet[MbdKey.kappa_tot_avg].head()

material_id
mp-1018100     [54.31892182143333]
mp-580941     [1.5558294449333332]
mp-2133        [49.68276724653333]
mp-1008559        [339.7319073431]
mp-8880        [79.33047527603334]
Name: kappa_tot_avg, dtype: object

In [43]:
df_pet_ref[MbdKey.kappa_tot_avg].head()

material_id
mp-1018100     [54.32141097156667]
mp-580941     [1.5540675070333334]
mp-2133        [49.62548450323334]
mp-1008559    [339.94404796423333]
mp-8880        [79.34418262496666]
Name: kappa_tot_avg, dtype: object

In [59]:
(df_pet[MbdKey.kappa_tot_avg] - df_pet_ref[MbdKey.kappa_tot_avg]).abs().sort_values(ascending=False).head(10)

material_id
mp-1541        [3.3050174859000094]
mp-20411        [1.887504484300024]
mp-984718       [1.409818217432985]
mp-10044       [0.5084684306998497]
mp-20012       [0.3967104247000002]
mp-1639        [0.2898121947000618]
mp-804         [0.2706433077666759]
mp-1008559    [0.21214062113335785]
mp-1479       [0.17240865880006595]
mp-20351      [0.16983644459999425]
Name: kappa_tot_avg, dtype: object

In [ ]:
df_our[MbdKey.kappa_tot_avg]

In [58]:
(df_pet[MbdKey.kappa_tot_avg] - df_our[MbdKey.kappa_tot_avg]).abs().sort_values(ascending=False).head(10)

material_id
mp-10044          [619.0730952229]
mp-1479       [175.52803036839995]
mp-984718     [123.52549262446644]
mp-1008559      [77.3967412125333]
mp-7631        [39.38473667406663]
mp-1639        [30.32622276500001]
mp-8882       [18.772546857866672]
mp-252        [17.802509409700015]
mp-20351      [17.315157255899997]
mp-1018100     [17.26434368879999]
Name: kappa_tot_avg, dtype: object

In [71]:
df_our['error_traceback']

material_id
mp-1018100      []
mp-580941     None
mp-2133         []
mp-1008559      []
mp-8880         []
              ... 
mp-1039         []
mp-1183441      []
no-mp-3         []
mp-1007661    None
mp-1018059    None
Name: error_traceback, Length: 103, dtype: object

# check calculator outputs

In [49]:
import ase
import numpy as np
from matbench_discovery.data import DataFilesCustomized as DataFiles
atoms_list = ase.io.read(DataFiles.phonondb_pbe_103_structures.path, index=":")

for i, atoms in enumerate(atoms_list):
    if atoms.info[Key.mat_id] == "mp-10044":
        print(i)
        break

from copy import deepcopy
atoms_pet = deepcopy(atoms)

from upet.calculator import UPETCalculator
from metatomic.torch.ase_calculator import MetatomicCalculator
from metatomic.torch import load_atomistic_model

calc_our = UPETCalculator(
            checkpoint_path="/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/checkpoints/pet-oam-xl-v1.0.0.ckpt",
            version="1.0.0", 
            device='cuda',
        )
model = load_atomistic_model(f"/mnt/shared-storage-gpfs2/lijiahang1/jobs/upet/checkpoints/pet-oam-xl-v1.0.0.pt")
calc_pet = MetatomicCalculator(model, device="cuda")

atoms.calc = calc_our
atoms_pet.calc = calc_pet

77


In [53]:
print("energy abs max %.4e" %
    np.abs(atoms.get_potential_energy() - atoms_pet.get_potential_energy()).max()
)
print("forces abs max %.4e" %
    np.abs(atoms.get_forces() - atoms_pet.get_forces()).max()
)
print("stress abs max %.4e" %
    np.abs(atoms.get_stress() - atoms_pet.get_stress()).max()
)

res = calc_pet.compute_energy(atoms, True)
print("energy abs max %.4e" %
    np.abs(res['energy'] - atoms_pet.get_potential_energy()).max()
)
print("forces abs max %.4e" %
    np.abs(res['forces'] - atoms_pet.get_forces()).max()
)
# print("stress abs max %.4e" %
#     np.abs(res['stress'] - atoms_pet.get_stress()).max()
# )



energy abs max 0.0000e+00
forces abs max 9.6112e-07
stress abs max 1.5832e-08


/mnt/shared-storage-user/lijiahang/miniconda3/envs/pet/lib/python3.12/site-packages/metatomic/torch/ase_calculator.py:1482: UserWarning: `compute_requested_neighbors_from_options` is deprecated and will be removed in a future version. Please use `neighbor_lists_for_model` to get the calculators and call them directly.
  vesin.metatomic.compute_requested_neighbors_from_options(


energy abs max 0.0000e+00
forces abs max 1.4231e-06


In [58]:
atoms.get_potential_energy()

np.float64(-45.50625991821289)

In [59]:
atoms.calc

In [ ]:
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE

atoms.set_constraint(FixSymmetry(atoms))

# Use standard mask for no-tilt constraint
filtered_atoms = FrechetCellFilter(
    atoms, mask=[True] * 3 + [False] * 3
)

optimizer = FIRE(
    filtered_atoms
)
optimizer.run(fmax=1e-4, steps=300)

/mnt/shared-storage-user/lijiahang/miniconda3/envs/pet/lib/python3.12/site-packages/metatomic_ase/_neighbors.py:78: UserWarning: `compute_requested_neighbors_from_options` is deprecated and will be removed in a future version. Please use `neighbor_lists_for_model` to get the calculators and call them directly.
  vesin.metatomic.compute_requested_neighbors_from_options(


      Step     Time          Energy          fmax
FIRE:    0 12:12:44      -45.504410        0.072312
FIRE:    1 12:12:44      -45.504562        0.069233
FIRE:    2 12:12:44      -45.504841        0.063220
FIRE:    3 12:12:44      -45.505203        0.054532
FIRE:    4 12:12:45      -45.505585        0.043548
FIRE:    5 12:12:45      -45.505928        0.030749
FIRE:    6 12:12:45      -45.506161        0.016683
FIRE:    7 12:12:45      -45.506260        0.001951
FIRE:    8 12:12:45      -45.506191        0.014311
FIRE:    9 12:12:45      -45.506191        0.014129
FIRE:   10 12:12:45      -45.506195        0.013768
FIRE:   11 12:12:45      -45.506199        0.013234
FIRE:   12 12:12:45      -45.506207        0.012531
FIRE:   13 12:12:45      -45.506207        0.011669
FIRE:   14 12:12:45      -45.506222        0.010659
FIRE:   15 12:12:45      -45.506226        0.009515
FIRE:   16 12:12:45      -45.506237        0.008107
FIRE:   17 12:12:45      -45.506248        0.006410
FIRE:   18 12:

np.True_

In [ ]:
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import FIRE

atoms_pet.set_constraint(FixSymmetry(atoms_pet))

# Use standard mask for no-tilt constraint
filtered_atoms_pet = FrechetCellFilter(
    atoms_pet, mask=[True] * 3 + [False] * 3
)

optimizer = FIRE(
    filtered_atoms_pet
)
optimizer.run(fmax=1e-4, steps=300)

/mnt/shared-storage-user/lijiahang/miniconda3/envs/pet/lib/python3.12/site-packages/metatomic/torch/ase_calculator.py:1482: UserWarning: `compute_requested_neighbors_from_options` is deprecated and will be removed in a future version. Please use `neighbor_lists_for_model` to get the calculators and call them directly.
  vesin.metatomic.compute_requested_neighbors_from_options(


      Step     Time          Energy          fmax
FIRE:    0 12:09:30      -45.504410        0.072312
FIRE:    1 12:09:30      -45.504562        0.069233
FIRE:    2 12:09:30      -45.504841        0.063220
FIRE:    3 12:09:30      -45.505203        0.054532
FIRE:    4 12:09:30      -45.505585        0.043548
FIRE:    5 12:09:30      -45.505924        0.030750
FIRE:    6 12:09:30      -45.506161        0.016683
FIRE:    7 12:09:30      -45.506260        0.001951
FIRE:    8 12:09:30      -45.506191        0.014311
FIRE:    9 12:09:30      -45.506191        0.014129
FIRE:   10 12:09:30      -45.506195        0.013768
FIRE:   11 12:09:30      -45.506199        0.013234
FIRE:   12 12:09:30      -45.506207        0.012531
FIRE:   13 12:09:30      -45.506210        0.011669
FIRE:   14 12:09:31      -45.506222        0.010659
FIRE:   15 12:09:31      -45.506226        0.009515
FIRE:   16 12:09:31      -45.506237        0.008108
FIRE:   17 12:09:31      -45.506248        0.006410
FIRE:   18 12:

np.True_

In [47]:
atoms_pet.calc

In [48]:
atoms.calc